# Week 3 — State Space, Eigenvalues, Modes, Poles

For the two-inertia plant, collect the states into

\[
x=
\begin{bmatrix}
\omega_1\\
\omega_2\\
\phi
\end{bmatrix}
\]

and write

\[
\dot x = Ax + Bu
\]

The eigenvalues of \(A\) are the plant's natural poles. Each eigenvalue corresponds to a mode with time dependence \(e^{\lambda t}\).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
J1, J2, k, c = 0.05, 0.08, 8.0, 0.15

A = np.array([
    [-c/J1,  c/J1, -k/J1],
    [ c/J2, -c/J2,  k/J2],
    [ 1.0,   -1.0,   0.0]
])

B = np.array([[1/J1], [0.0], [0.0]])
C = np.array([[1.0, 0.0, 0.0]])

eigvals, eigvecs = np.linalg.eig(A)
eigvals

array([-2.43750000e+00+15.93921559j, -2.43750000e+00-15.93921559j,
        1.65988756e-15 +0.j        ])

A complex pair

\[
\lambda=\sigma\pm j\omega_d
\]

means **oscillation plus decay**:

- \(\sigma<0\): envelope decays
- \(\omega_d\): oscillation frequency

So there is no contradiction between “eigenvalues tell me decay rate” and “eigenvalues describe vibration modes.” A complex eigenvalue tells you both at once.

In [3]:
for i, lam in enumerate(eigvals):
    print(f"Mode {i+1}: λ = {lam:.4f}")
    if abs(lam.imag) > 1e-8:
        print(f"  decay rate = {lam.real:.4f} 1/s")
        print(f"  damped frequency = {abs(lam.imag):.4f} rad/s")

Mode 1: λ = -2.4375+15.9392j
  decay rate = -2.4375 1/s
  damped frequency = 15.9392 rad/s
Mode 2: λ = -2.4375-15.9392j
  decay rate = -2.4375 1/s
  damped frequency = 15.9392 rad/s
Mode 3: λ = 0.0000+0.0000j


## Controllability

A mode is useful to know about, but can the input actually move it?

For an \(n\)-state system, form

\[
\mathcal C = [B,\ AB,\ A^2B,\ldots,A^{n-1}B]
\]

Full rank means every state-space direction is controllable.

In [4]:
Ctrb = np.hstack([B, A@B, A@A@B])
print("Controllability rank:", np.linalg.matrix_rank(Ctrb), "of", A.shape[0])

Obsv = np.vstack([C, C@A, C@A@A])
print("Observability rank:", np.linalg.matrix_rank(Obsv), "of", A.shape[0])

Controllability rank: 3 of 3
Observability rank: 3 of 3


## The physical interpretation

A mode is always part of the model, but its coefficient can be zero for a particular initial condition/input.

“Exciting a mode” means the input or initial condition gives that modal component a nonzero amplitude.

That is the connection to the constant \(c_i\) in:

\[
x(t)=\sum_i c_i v_i e^{\lambda_i t}
\]

The eigenvalue/eigenvector defines **what the mode is**. The coefficient \(c_i\) defines **how much of it is present in this response**.